## Notes

In [ ]:
# can travel for 1/4 sq km in 1hr - 1 flight
# each sq km is 18,000rs

In [ ]:
INDIA_PROJECTED_CRS = "24378"

## Setup

In [1]:
# set libraries to refresh
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import openpyxl

import matplotlib.pyplot as plt

# import kml reading and set supported driver
import fiona

fiona.drvsupport.supported_drivers["KML"] = "rw"

In [3]:
from gridsample.utils import save_shapefiles
# from gridsample.mapping.plot import create_interactive_map

In [4]:
ROOT_DIR = Path("../")
DATA_DIR = ROOT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "00_raw"
OUTPUT_DATA_DIR = DATA_DIR / "01_processed" / "Goverment Buildings"
OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)

CONSOLIDATED_DATA_PATH = RAW_DATA_DIR / "government_buildings" / "Consolidated_data_09042025.xlsx"

In [5]:
workbook = openpyxl.load_workbook(CONSOLIDATED_DATA_PATH)


## Utilities

In [6]:
def is_cell_highlighted(cell):
    if cell.fill.fill_type == 'solid':
        # Check if the cell has a background color (not white/none)
        if cell.fill.start_color.rgb != '00000000' and cell.fill.start_color.rgb != 'FFFFFFFF':
            return True
    return False


def get_highlighted_rows(worksheet):
    highlighted_rows = []

    for row_idx, row in enumerate(worksheet.iter_rows(min_row=2)):  # Skip header row
        if sum(is_cell_highlighted(cell) for cell in row) >= 24:
            highlighted_rows.append(row_idx)
    
    return highlighted_rows

## Seoni

In [7]:
SAVE_DIR_SEONI = OUTPUT_DATA_DIR / "Seoni_new_09042025/"
SAVE_DIR_SEONI.mkdir(parents=True, exist_ok=True)


worksheet_seoni = workbook["Seoni"]

df_seoni = pd.read_excel(CONSOLIDATED_DATA_PATH, sheet_name="Seoni", header=0)
df_seoni

,SN,District,DISCOM,Name of institutions/consumer as per billing/connection details,Contact information of consumer,govt_dept_name,IVRS/Consumer Code as per billing details,Latitude of location of consumer's permises,longitude of location of consumer's permises,Owner ship a of primises (Govt./ Private),...,Contract Demand/ sanctioned load of consumer for target premises (KW)),Rated capacity of DTR as per records of Discom (kVA),How much total solar rooftop capacity alredy connected with/ applied for at concerned DTr as per records of Discom (in kW),Possible solar rooftop capacity (in kW) as per Contract Demand (Option -1),Possible solar rooftop capacity (in kW) as per shadow free useful rooftop/groun d area of consumer's premises (Option -2),Possible solar rooftop capacity (in kW) as per monthly consumption (Option -3),Possible solar rooftop capacity (in kW) as per DTR Capacity for solar rooftop (Option -4),"Proposed SPV Plant Capacity (kW) = minimum of C-1, C-2, C-3 and C-4","Critical obseravation/ remarks, if any",Feasibility analysis as per proposed SPV plat capacity and rooftop space availability
0,1,SEONI,MPPKV VCL,"AE HOUSING BOARD JBP, CHHIDIYA PALARI, TRIBAL ...",7.987507e+09,Tribal Research and Development Institute (TRDI),1313046966,22.084803,79.594453,Govt.,...,53.04,100,0,53.0,64.58,66.25,72.0,53.00,NaN,Feasible
1,2,SEONI,MPPKV VCL,"ASSISTENT ENGINEER TESTING, 132 KV SUB STATION...",9.425803e+09,Transmission company,1412006481,22.102893,79.557736,Govt.,...,65.04,200,0,65.0,79.20,92.69,144.0,65.00,NaN,Feasible
2,3,SEONI,MPPKV VCL,"A E MPPKVV CO LTD, 132 KV SUBSTATION, SC 89279...",NaN,Transmission company,1121014415,22.620690,79.631206,Govt.,...,50.00,25,0,50.0,60.92,37.40,18.0,18.00,NaN,Feasible
3,4,SEONI,MPPKV VCL,"A.E. 220 KV SUB STATION, SIMARIYA, MPPKVV CO.L...",9.425805e+09,Transmission company,1313018680,22.141547,79.525387,Govt.,...,50.00,25,0,50.0,60.92,106.97,18.0,18.00,NaN,Feasible
4,5,SEONI,MPPKV VCL,"PRINCIPAL I T I, S C NO. 10701, INDUSTRIAL TEC...",9.770403e+09,Technical Education and Skill Development,1313039188,22.086900,79.595713,Govt.,...,74.00,200,0,74.0,90.17,9.97,144.0,9.97,NaN,Feasible
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68,69,SEONI,MPPKV VCL,"THE SDO TELEPHONE SEONI, KANHIWARA EXCHANGE, B...",9.422939e+09,BSNL,1318015853,22.207170,79.742583,Govt.,...,27.04,100,0,27.0,32.90,6.23,72.0,6.23,NaN,Feasible
69,70,SEONI,MPPKV VCL,"SHRI S.D.O. BHARAT SANCHAR, NIGAM, CHHAPARA LA...",9.425731e+09,BSNL,1140008699,22.398460,79.546297,Govt.,...,25.04,100,0,25.0,30.46,17.71,72.0,17.71,NaN,Feasible
70,71,SEONI,MPPKV VCL,"S.D.O. TELEGRAPH, BHARAT SANCHAR NIGAM LTD., B...",9.422939e+09,BSNL,1319006784,22.307994,79.813616,Govt.,...,25.04,100,0,25.0,30.46,6.25,72.0,6.25,NaN,Feasible
71,72,SEONI,MPPKV VCL,"SDO PHONS SEONI, TELEPHONE EXCHANGE BARAPATTHA...",9.425001e+09,BSNL,1412022735,22.101222,79.550589,Govt.,...,25.04,100,0,25.0,30.46,23.48,72.0,23.48,NaN,Feasible


In [ ]:

rows_to_drop_seoni = get_highlighted_rows(worksheet_seoni)

filtered_rows_df_seoni = df_seoni.drop(index=rows_to_drop_seoni).reset_index(drop=True)

# Filter to only building names and lat / long
filtered_df_seoni = filtered_rows_df_seoni[["Name of institutions/consumer as per billing/connection details", 
                 "Latitude of location of consumer's permises",
                 "longitude of location of consumer's permises",]]
filtered_df_seoni.rename(columns={
    "Name of institutions/consumer as per billing/connection details": "Name",
    "Latitude of location of consumer's permises": "lat",
    "longitude of location of consumer's permises": "lon"
}, inplace=True)

filtered_df_seoni.dropna().reset_index(drop=True, inplace=True)

filtered_df_seoni.to_csv(SAVE_DIR_SEONI / "cleaned_data.csv", index=False)

/var/folders/mq/6zyh6j6j1wzc58dyc13z2hr00000gn/T/ipykernel_9430/2578903169.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df_seoni.rename(columns={


## Dhar

In [9]:
SAVE_DIR_DHAR = OUTPUT_DATA_DIR / "Dhar/"
SAVE_DIR_DHAR.mkdir(parents=True, exist_ok=True)

worksheet_dhar = workbook["Dhar"]

df_dhar = pd.read_excel(CONSOLIDATED_DATA_PATH, sheet_name="Dhar", header=0)
df_dhar

,"S,No.",Division,Consumer No.,DTR Capacity,Sanctioned Load (kW),Consumer Name (Name of Office),govt_dept_name,Address,Mobile No.,Average Consumption (kwh),Net Bill,Types of Building,Types of Rooftop RCC/Non RCC,Shadow free area (Sq. Ft.),Solar Rooftop Capacity installed/proposed (In kWp),Latitude & Longitude of premises
0,1,2,3,4,5,6,NaN,7,8.000000e+00,NaN,9,10,11,12,13.0,14
1,1,Badnawar,3783009255,63,10,तहसीलदार तहसील ऑफिस,Revenue Department,बड़ी चौपाटी,7.295232e+09,NaN,5910,Pakka,RCC,2400,10.0,"23.022311, 75.2475604.17"
2,2,Badnawar,3783009291,200,105,मेडिकल आफिसर पब्लिक हेल्थ सेन्टर,Public Health & Family Welfare Department,बस स्टेण्ड,9.893875e+09,7041.0,153322,Pakka,RCC,8000,105.0,"23.0218737, 75.2323902.17"
3,3,Badnawar,3783010564,63,50,सचिव कृषि उपज मंण्डी,Farmers Welfare and Agricultural Development D...,बदनावर शहर,7.722992e+09,104214.0,137196,Pakka,RCC,3000,42.0,"23.0182396, 75.2024675.1425"
4,4,Badnawar,3783011288,25,10,ग्रामीण् कृषि विस्तार अधिकारी,Farmers Welfare and Agricultural Development D...,मिटटी परीक्षण प्रयोगशाला,8.120006e+09,NaN,30754,Pakka,RCC,1600,10.0,"23.0270422, 75.2362377"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
174,174,Mhow,3689004141,100,10,तहसीलदार महोदया,Revenue Department,मॉडर्न रिकॉर्ड रूम,9.111113e+09,NaN,69806,Govt.,RCC,1800,10.0,"22.152777, 75.351798"
175,175,Mhow,3689009686,200,12,तहसीलदार तहसील ऑफिस,Revenue Department,धरमपुरी,9.893353e+09,NaN,148913,Govt.,RCC,3600,12.0,22.152756. 75.351673
176,176,Mhow,3606012684,25,10,मेडिकल ऑफिसर,Public Health & Family Welfare Department,प्राथमिक स्वास्थ्य केंद्र गुजरी,8.224029e+09,NaN,57957,Govt.,RCC,2000,10.0,"22.31876, 75.5056"
177,177,Dhar (O&M),H5074904000,315,80,प्राचार्य पी जी कॉलेज धार,Higher Education,नंबर 62 जेतपुर अहमदाबाद इंदौर रोड धार,9.425936e+09,NaN,127925,Govt.,RCC,4000,80.0,"22.6097, 75.3245"


In [10]:
rows_to_drop = get_highlighted_rows(worksheet_dhar)

filtered_rows_df_dhar = df_dhar.drop(index=rows_to_drop + [0] # drop the first row which seems to be meaningless
                           ).reset_index(drop=True)
filtered_df_dhar = filtered_rows_df_dhar[['Consumer Name  (Name of Office)',
                                "Latitude & Longitude of premises"]]
filtered_df_dhar

,Consumer Name (Name of Office),Latitude & Longitude of premises
0,तहसीलदार तहसील ऑफिस,"23.022311, 75.2475604.17"
1,मेडिकल आफिसर पब्लिक हेल्थ सेन्टर,"23.0218737, 75.2323902.17"
2,ग्रामीण् कृषि विस्तार अधिकारी,"23.0270422, 75.2362377"
3,प्राचार्या शासकीय महावि‌द्यालय बदनावर,"23.0268754, 75.2273352.21"
4,प्राचार्या शासकीय कालेज कानवन,"22.857444, 75.249566"
...,...,...
162,तहसीलदार महोदया,"22.152777, 75.351798"
163,तहसीलदार तहसील ऑफिस,22.152756. 75.351673
164,मेडिकल ऑफिसर,"22.31876, 75.5056"
165,प्राचार्य पी जी कॉलेज धार,"22.6097, 75.3245"


In [ ]:
# First split by commas or ". " to get lats
filtered_df_dhar["lat"] = filtered_df_dhar["Latitude & Longitude of premises"].apply(lambda x: str(x).split(",")[0].split(". ")[0])

# Then split by "." because some numbers have double decimal points -_-
filtered_df_dhar["lat"] = filtered_df_dhar["lat"].apply(lambda x: x 
                                              if len(x.split(".")) <= 2 
                                              else x.split(".")[0] + "." + x.split(".")[1]
                                              ).astype(float)

# INSPECT MANUALLY BECAUSE THERE MAY STILL BE SOME INSANITY
filtered_df_dhar["lat"].tolist()


/var/folders/mq/6zyh6j6j1wzc58dyc13z2hr00000gn/T/ipykernel_9430/1759762080.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df_dhar["lat"] = filtered_df_dhar["Latitude & Longitude of premises"].apply(lambda x: str(x).split(",")[0].split(". ")[0])
/var/folders/mq/6zyh6j6j1wzc58dyc13z2hr00000gn/T/ipykernel_9430/1759762080.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df_dhar["lat"] = filtered_df_dhar["lat"].apply(lambda x: x


[23.022311,
 23.0218737,
 23.0270422,
 23.0268754,
 22.857444,
 22.869749,
 22.859749,
 22.365676,
 22.373413,
 22.373413,
 22.376394,
 22.376577,
 22.509078,
 22.113214,
 22.112493,
 22.10439,
 22.119886,
 22.119886,
 22.5499492,
 22.614715,
 22.617751,
 22.61515,
 22.612195,
 22.592665,
 22.588172,
 22.606041,
 22.60966,
 22.611647,
 22.5921156,
 22.603721,
 22.603721,
 22.608513,
 22.603404,
 22.592686,
 22.607712,
 22.604026,
 22.5826,
 22.597801,
 22.591135,
 22.576629,
 22.576898,
 22.590376,
 22.592171,
 22.609995,
 22.59997,
 22.601903,
 22.612346,
 22.61113,
 22.61159115,
 22.592817,
 22.610276,
 22.609997,
 22.601511,
 22.601418,
 22.607053,
 22.61007,
 22.597207,
 22.60686,
 22.598913,
 22.613399,
 22.3493691,
 22.35021888,
 22.3488652,
 22.34526,
 22.3472992,
 22.3476307,
 22.3477895,
 22.3477895,
 22.3406615,
 22.205109,
 22.208423,
 22.7222,
 22.219053,
 22.201674,
 22.205352,
 22.207793,
 22.205215,
 22.22441,
 22.293419,
 22.2069274,
 22.2077529,
 22.2388651,
 22.226219

In [12]:
# Rinse and repeat for lons
filtered_df_dhar["lon"] = filtered_df_dhar["Latitude & Longitude of premises"].apply(lambda x: str(x).split(", ")[-1])

# Look for specific cases of outliers and fix:
pattern1 = r'(\d{2}\.\d+,\d{2}\.\d+|\d{2}\.\d+\.\d{2}\.\d+)'
pattern1_matches = filtered_df_dhar.loc[filtered_df_dhar["lon"].str.contains(pattern1, regex=True, na=False), "lon"]
filtered_df_dhar.loc[pattern1_matches.index, "lon"] = pattern1_matches.apply(lambda x: str(x).split(",")[-1])

pattern2 = r'(\d{2}\.\d+. \d{2}\.\d+|\d{2}\.\d+\.\d{2}\.\d+)'
pattern2_matches = filtered_df_dhar.loc[filtered_df_dhar["lon"].str.contains(pattern2, regex=True, na=False), "lon"]
filtered_df_dhar.loc[pattern2_matches.index, "lon"] = pattern2_matches.apply(lambda x: str(x).split(". ")[-1])


pattern3 = r'(\d{2}\.\d+\.\d+)'
pattern3_matches = filtered_df_dhar.loc[filtered_df_dhar["lon"].str.contains(pattern3, regex=True, na=False), "lon"]
filtered_df_dhar.loc[pattern3_matches.index, "lon"] = pattern3_matches.apply(lambda x: str(x).split(".")[0] + "." + str(x).split(".")[1])

pattern4 = r'^[^.\,]*$'
not_nans = filtered_df_dhar["lon"] != "nan"
pattern4_matches = filtered_df_dhar.loc[(not_nans) & (filtered_df_dhar["lon"].str.contains(pattern4, regex=True, na=False)), "lon"]
filtered_df_dhar.loc[pattern4_matches.index, "lon"] = pattern4_matches.apply(lambda x: str(x)[:2] + "." + str(x)[2:])

pattern5 = r'(\d{2}\,\d+)'
pattern5_matches = filtered_df_dhar.loc[filtered_df_dhar["lon"].str.contains(pattern5, regex=True, na=False), "lon"]
filtered_df_dhar.loc[pattern5_matches.index, "lon"] = pattern5_matches.apply(lambda x: str(x).replace(",", "."))


filtered_df_dhar.loc[filtered_df_dhar["lon"] == '  75. ....3224', 'lon'] = '75.3224'

# convert to float
filtered_df_dhar["lon"] = filtered_df_dhar["lon"].astype(float)

# INSPECT MANUALLY BECAUSE THERE MAY STILL BE SOME INSANITY
filtered_df_dhar["lon"].tolist()

/var/folders/mq/6zyh6j6j1wzc58dyc13z2hr00000gn/T/ipykernel_9430/2615866732.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df_dhar["lon"] = filtered_df_dhar["Latitude & Longitude of premises"].apply(lambda x: str(x).split(", ")[-1])
/var/folders/mq/6zyh6j6j1wzc58dyc13z2hr00000gn/T/ipykernel_9430/2615866732.py:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  pattern1_matches = filtered_df_dhar.loc[filtered_df_dhar["lon"].str.contains(pattern1, regex=True, na=False), "lon"]
/var/folders/mq/6zyh6j6j1wzc58dyc13z2hr00000gn/T/ipykernel_9430/2615866732.py:10: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actu

[75.2475604,
 75.2323902,
 75.2362377,
 75.2273352,
 75.249566,
 75.255457,
 75.255457,
 74.785912,
 74.786209,
 74.786209,
 74.784848,
 74.785263,
 74.849833,
 74.585403,
 74.592212,
 74.599692,
 74.585245,
 74.585245,
 75.2801073,
 75.33000078,
 75.3300078,
 75.3300078,
 75.32592,
 75.3111253,
 75.2777,
 75.3168,
 75.3122,
 75.3149,
 75.312831,
 75.32096,
 75.320968,
 75.3308,
 75.32387,
 75.312154,
 75.32246,
 75.32246,
 75.34558,
 75.3243,
 75.332258,
 75.3444,
 75.35467,
 75.32308,
 75.328533,
 75.31216,
 75.323576,
 75.32311,
 75.313464,
 75.310641,
 75.332356,
 75.32308,
 75.33,
 75.31234,
 75.3353319,
 75.334456,
 75.321153,
 75.31246,
 75.295763,
 75.325328,
 75.32463,
 75.324058,
 75.90306,
 74.9997464,
 74.9986953,
 74.96917,
 75.0029017,
 75.0044725,
 75.0069713,
 75.005426,
 75.0070511,
 74.755039,
 74.698488,
 74.759126,
 74.756788,
 74.756706,
 74.754776,
 74.755039,
 74.75535,
 74.757647,
 74.883717,
 75.0996808,
 75.1021272,
 75.0908471,
 75.0818157,
 75.0842305,
 75.3

In [19]:
filtered_df_dhar.drop(
    columns=["Latitude & Longitude of premises"]
    ).dropna(
        axis=0
    ).rename(
        columns={"Consumer Name  (Name of Office)": "Name"}
    ).to_csv(SAVE_DIR_DHAR / "cleaned_data.csv", index=False)

## Chhindwara

In [20]:
SAVE_DIR_CHHINDWARA = OUTPUT_DATA_DIR / "Chhindwara/"
SAVE_DIR_CHHINDWARA.mkdir(parents=True, exist_ok=True)

worksheet_chhindwara = workbook["Chhindwara"]

df_chhindwara = pd.read_excel(CONSOLIDATED_DATA_PATH, sheet_name="Chhindwara", header=0)
df_chhindwara

,SN,District,DISCOM,Name of institutions/consumer as per billing/connection details,Contact information of consumer,govt_dept_name,IVRS/Consumer Code as per billing details,Latitude/Langitude of location of consumer's permises,Owner ship a of primises (Govt./ Private),Average monthly electricity consumption on of consumer's premises (kWh),...,Contract Demand/ sanctioned load of consumer for target premises (kVA),Contract Demand/ sanctioned load of consumer for target premises (KW),Rated capacity of DTR as per records of Discom (kVA),How much total solar rooftop capacity alredy connected with/ applied for at concerned DTr as per records of Discom (in kW),Possible solar rooftop capacity (in kW) as per Contract Demand (Option -1),Possible solar rooftop capacity (in kW) as per shadow free useful rooftop/groun d area of consumer's premises (Option -2),Possible solar rooftop capacity (in kW) as per monthly consumption (Option -3),Possible solar rooftop capacity (in kW) as per DTR Capacity for solar rooftop (Option -4),"Proposed SPV Plant Capacity (kW) = minimum of C-1, C-2, C-3 and C-4","Critical obseravation/ remarks, if any"
0,1,CHHINDWARA,JABALPUR,S D O TELEPHONE EXCHANGE,,BSNL,1342016220,2.2906834 79.165359,GOVT,2024.67,...,35.0,NaN,63.0,0.0,35.0,20.0,13.50,50.4,13.50,NaN
1,2,CHHINDWARA,JABALPUR,BRANCH MANEGER,NaN,-,1342018607,2.3032795 79.170153,GOVT,1058.80,...,24.0,NaN,200.0,0.0,24.0,20.0,7.06,160.0,7.06,NaN
2,3,CHHINDWARA,JABALPUR,PRINCIPAL GOVT PG COLLAGE,NaN,Department of Higher Education,1342022971,2.3050235 79.152663,GOVT,522.79,...,20.0,20.0,63.0,0.0,20.0,20.0,3.49,50.4,3.49,NaN
3,4,CHHINDWARA,JABALPUR,BLOCK MEDICAL OFFICER,NaN,Public Health & Family Welfare Department,1342017750,2.3062417 79.171475,GOVT,3098.42,...,50.0,50.0,200.0,0.0,50.0,20.0,20.66,160.0,20.00,NaN
4,5,CHHINDWARA,JABALPUR,SHRI NEW CIVIL COURT,9303107125,Law & Legislative Affairs Department,1342000356,2.3059878 79.168829,GOVT,1782.43,...,37.5,30.0,100.0,0.0,37.5,6.0,11.88,80.0,6.00,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
153,154,CHHINDWARA,JABALPUR,GENERAL MANAGER W C F LTD,9406776775,NaN,H4991832000,2.1981319 78.684302,GOVT,154000.00,...,800.0,640.0,1000.0,0.0,800.0,100.0,1026.67,800.0,100.00,NaN
154,155,CHHINDWARA,JABALPUR,GENERAL MANAGER W C F LTD,9406776775,NaN,H7203832000,22..189149 78.705791,GOVT,287000.00,...,1500.0,1200.0,3150.0,0.0,1500.0,400.0,1913.33,2520.0,400.00,NaN
155,156,CHHINDWARA,JABALPUR,MANAGER MOHAN COLLIERY,7587595224,Coal department,H8203832000,22.178109 78.677624,GOVT,243000.00,...,1400.0,1120.0,3150.0,0.0,1400.0,200.0,1620.00,2520.0,200.00,NaN
156,157,CHHINDWARA,JABALPUR,EXECUTIVE ENGINEER PHE,9630526057,Public Health Engineering Department,H8282832000,22.226721 78.687877,GOVT,94400.00,...,400.0,320.0,500.0,0.0,400.0,100.0,629.33,400.0,100.00,NaN


In [23]:
rows_to_drop = get_highlighted_rows(worksheet_chhindwara)

filtered_rows_df_chhindwara = df_chhindwara.drop(index=rows_to_drop
                           ).reset_index(drop=True)
filtered_df_chhindwara = filtered_rows_df_chhindwara[[
    'Name of institutions/consumer as per billing/connection details',
    "Latitude/Langitude of location of consumer's permises"]]
filtered_df_chhindwara

,Name of institutions/consumer as per billing/connection details,Latitude/Langitude of location of consumer's permises
0,PRINCIPAL GOVT PG COLLAGE,2.3050235 79.152663
1,BLOCK MEDICAL OFFICER,2.3062417 79.171475
2,SHRI NEW CIVIL COURT,2.3059878 79.168829
3,BLOCK MEDICAL OFFICER PLANT,22.308125 79.171005
4,GOVT ITI BHAVAN HARANBHATA,0
...,...,...
116,EE PHE DN. MANDHAN DAM\nDARBAI,NaN
117,THE ACCOUNTS\nOFFICERW.C.F.LTD,22.1919 78.4664
118,"CMO NAGAR PALIKA PARISHAD ,\nPARASIA",22.202985 78.747832
119,EXECUTIVE ENGINEER PHE,22.226721 78.687877


In [26]:
filtered_df_chhindwara["Latitude/Langitude of location of consumer\'s permises"].to_list()
filtered_df_chhindwara["lat"] = filtered_df_chhindwara[
    "Latitude/Langitude of location of consumer\'s permises"
    ].apply(lambda x: str(x).split(" ")[0])
filtered_df_chhindwara["lat"].to_list()

/var/folders/mq/6zyh6j6j1wzc58dyc13z2hr00000gn/T/ipykernel_9430/906388244.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df_chhindwara["lat"] = filtered_df_chhindwara[


['2.3050235',
 '2.3062417',
 '2.3059878',
 '22.308125',
 '0',
 '2.3885238',
 '2.6143151',
 '2.6102309',
 '2.6162435',
 '22.5851556',
 '2.6239426',
 '2.2061665',
 '2.2008466',
 '2.1530953',
 '2.0747527',
 '2.0738473',
 '22.0668819',
 '2.0658762',
 '2.0549078',
 '2.0718852',
 '2.0612681',
 '2.0719148',
 '2.0595355',
 '2.0714561',
 '2.0706144',
 '2.0554717',
 '2.0679167',
 '2.0679083',
 '2.0499185',
 '2.0498204',
 '2.0768912',
 '22.068885',
 '2.0705775',
 '2.0536433',
 '2.0649162',
 '2.0518697',
 '22.0739475',
 '22.0737497',
 '2.1017009',
 '2.0971903',
 '2.0281998',
 '2.0708245',
 '22.0745626',
 '2.0598074',
 '2.0539026',
 '2.0694375',
 '2.0755556',
 '2.0543365',
 '2.0624912',
 '22.0546336',
 '22.075188',
 '2.0548334',
 '53.24156',
 '22.066401',
 '22.0571326',
 '22.0613096',
 '22.0651783',
 '21.964283',
 '2.0005865',
 'nan',
 '1.8900983',
 '2.0785821',
 '2.0790884',
 '2.0622168',
 '21.858666',
 '1.8148777',
 '1.8138341',
 '21.94098',
 '2.0557837',
 '22.039025',
 '1.8036615',
 '22.195722',